# Block Matrix Multiplication

Chapter 10: Matrix Block Partitioning · *Mathematics for AI and Machine Learning* by Fuheng Wu

Verify block-product rules against elementwise multiplication results.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-10/block-multiplication/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's default 2x2 example: A = [[a, b], [c, d]] and B = [[d, c], [b, a]]
a, b, c, d = 1.2, 0.6, -0.7, 1.5
A = torch.tensor([[a, b], [c, d]], dtype=torch.float64)
B = torch.tensor([[d, c], [b, a]], dtype=torch.float64)
C = A @ B
print("C = A B =", C.tolist())
print("C11 = a d + b b =", a * d + b * b, " C12 = a c + b a =", a * c + b * a)

# ── The same rule with blocks: each entry becomes a matrix ──────────────────
n = 3                                         # blocks of size 3x3 -> 6x6 matrices
A_big = torch.randn(2 * n, 2 * n, dtype=torch.float64)
B_big = torch.randn(2 * n, 2 * n, dtype=torch.float64)


def blocks(M):
    return M[:n, :n], M[:n, n:], M[n:, :n], M[n:, n:]


A11, A12, A21, A22 = blocks(A_big)
B11, B12, B21, B22 = blocks(B_big)
C_blocks = torch.cat([
    torch.cat([A11 @ B11 + A12 @ B21, A11 @ B12 + A12 @ B22], dim=1),
    torch.cat([A21 @ B11 + A22 @ B21, A21 @ B12 + A22 @ B22], dim=1),
], dim=0)
print("\nblock product == full product:", torch.allclose(C_blocks, A_big @ B_big))

# Order matters inside blocks: A12 @ B21 is not B21 @ A12
print("A12 B21 == B21 A12 ?", torch.allclose(A12 @ B21, B21 @ A12))

# ── Block-diagonal matrices multiply block by block ─────────────────────────
D1, D2 = torch.block_diag(A11, A22), torch.block_diag(B11, B22)
print("block_diag product == block_diag of products:",
      torch.allclose(D1 @ D2, torch.block_diag(A11 @ B11, A22 @ B22)))

C = A B = [[2.1599999999999997, -0.12], [-0.1499999999999999, 2.2899999999999996]]
C11 = a d + b b = 2.1599999999999997  C12 = a c + b a = -0.12

block product == full product: True
A12 B21 == B21 A12 ? False
block_diag product == block_diag of products: True
